In [7]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cdsapi
import copernicusmarine

c:\Users\aleks\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Let us read the data:

In [2]:
file_name = "PositionReport.xlsx"
df = pd.read_excel(file_name)
print("\nFound columns:")
print(df.columns.tolist())

print("\nFirst 10 rows of the data:")
print(df.head(10))


Found columns:
['VESSEL', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']

First 10 rows of the data:
                           VESSEL      Unnamed: 1       Unnamed: 2  \
0                             NaN             NaN              NaN   
1                            From             NaN              NaN   
2                              To             NaN              NaN   
3                             NaN             NaN              NaN   
4    Total miles (w/speed ≥ 1 kn)             NaN              NaN   
5  Average speed (w/speed ≥ 5 kn)             NaN              NaN   
6                             NaN             NaN              NaN   
7                            Time             Lat              Lon   
8                             NaN             NaN              NaN   
9             2023-02-02 00:06:00  56° 49.6000' N  000° 07.7000' W   

            Unnamed: 3               Unnamed: 4   Unnamed: 5  \
0           

What can be seen in the data is the fact that the header starts at row 9. Therefore, first 8 rows should be skipped. 

In [18]:
df_position = pd.read_excel('PositionReport.xlsx', sheet_name='SC CONNECTOR', skiprows=8)

In [19]:
print("Data shape:", df_position.shape)
display(df_position.head(10))

Data shape: (156943, 8)


,Time,Lat,Lon,Speed\n[kn],Calculated\nSpeed\n[kn],Course\n[°],Distance since\nlast point\n[nm],Total distance\nrun\n[nm]
0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000
2,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126
3,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728
4,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653
5,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580
6,2023-02-02 01:33:00,57° 06.3000' N,000° 16.8000' E,16.0,15.9,48.0,4.763985,22.265565
7,2023-02-02 01:45:50,57° 08.4690' N,000° 21.3304' E,16.0,15.4,48.0,3.285002,25.550568
8,2023-02-02 01:58:32,57° 10.7088' N,000° 25.9707' E,16.2,15.9,47.0,3.374881,28.925449
9,2023-02-02 02:10:15,57° 12.6930' N,000° 30.4486' E,15.9,16.1,50.0,3.139759,32.065207


In [20]:
print("Data shape before cleaning:", df_position.shape)

df_position = df_position.dropna(subset=['Time']).reset_index(drop=True)

df_position.columns = df_position.columns.str.replace('\n', ' ')

# standarization of the column names standarization
rename_dict = {'Speed [kn]': 'Speed_kn', 'Calculated Speed [kn]': 'Calc_Speed_kn', 'Course [°]': 'Course_deg', 'Distance since last point [nm]': 'Dist_since_last_nm', 'Total distance run [nm]': 'Total_dist_nm'}

df_position = df_position.rename(columns=rename_dict)

print("Data shape after cleaning", df_position.shape)
display(df_position.head(10))

Data shape before cleaning: (156943, 8)
Data shape after cleaning (156942, 8)


,Time,Lat,Lon,Speed_kn,Calc_Speed_kn,Course_deg,Dist_since_last_nm,Total_dist_nm
0,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000
1,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126
2,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728
3,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653
4,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580
5,2023-02-02 01:33:00,57° 06.3000' N,000° 16.8000' E,16.0,15.9,48.0,4.763985,22.265565
6,2023-02-02 01:45:50,57° 08.4690' N,000° 21.3304' E,16.0,15.4,48.0,3.285002,25.550568
7,2023-02-02 01:58:32,57° 10.7088' N,000° 25.9707' E,16.2,15.9,47.0,3.374881,28.925449
8,2023-02-02 02:10:15,57° 12.6930' N,000° 30.4486' E,15.9,16.1,50.0,3.139759,32.065207
9,2023-02-02 02:24:00,57° 15.1000' N,000° 35.9000' E,16.0,16.6,50.0,3.814630,35.879838


Apart from the 'nice formatting' of the Excel file, there is another aspect to consider. From human perspective, the 56° 49.6000' N notation is highly readable. Howeber, for a computer this notation is treated as a string not a particular number. Therefore, we need to tranform it into a floating-point number, 56.8266.

In [21]:
def parse_coordinates(coord_str):
    """
    Changing "56° 49.6000' N" string format for a decimal (float) format.
    """

    if pd.isna(coord_str):
        return np.nan

    match = re.search(r"(\d+)\s*°\s*([\d\.]+)\s*'\s*([NSEW])", str(coord_str))

    if match:
        degrees = float(match.group(1))
        minutes = float(match.group(2))
        direction = match.group(3)

        decimal = degrees + (minutes / 60.0)

        if direction in ['S', 'W']:
            decimal = -decimal

        return decimal
    return np.nan


df_position['Lat_dec'] = df_position['Lat'].apply(parse_coordinates)
df_position['Lon_dec'] = df_position['Lon'].apply(parse_coordinates)

display(df_position[['Lat', 'Lat_dec', 'Lon', 'Lon_dec']].head()) # comparing all columns - old ones - with the old notation and new ones - with the new notation

,Lat,Lat_dec,Lon,Lon_dec
0,56° 49.6000' N,56.826667,000° 07.7000' W,-0.128333
1,56° 53.2000' N,56.886667,000° 07.5000' W,-0.125000
2,56° 55.4000' N,56.923333,000° 03.5000' W,-0.058333
3,56° 59.2000' N,56.986667,000° 03.3000' E,0.055000
4,57° 03.0000' N,57.050000,000° 10.5000' E,0.175000


In [22]:
df_position = df_position.drop(columns=['Lat', 'Lon'])
df_position = df_position.rename(columns={'Lat_dec': 'Lat', 'Lon_dec': 'Lon'})

order = [
    'Time', 'Lat', 'Lon', 'Speed_kn', 'Calc_Speed_kn', 'Course_deg',
    'Dist_since_last_nm', 'Total_dist_nm'
]

df_position = df_position[order]

print("Columns after dropping old and renaming new ones:")
print(df_position.columns.tolist())

df_position.to_csv('PositionReport_cleaned.csv', index=False)

Columns after dropping old and renaming new ones:
['Time', 'Lat', 'Lon', 'Speed_kn', 'Calc_Speed_kn', 'Course_deg', 'Dist_since_last_nm', 'Total_dist_nm']


Now, we need to analyze the bounding box for the SC CONNECTOR - the furthest points North, South, East and West that the ship reached and the years in which it sailed.

In [24]:
df = pd.read_csv('PositionReport_cleaned.csv')

# time conversion (in datetime format)
df['Time'] = pd.to_datetime(df['Time']) 

# bounding box with some margin (0.5 degrees)
margin = 0.5 
north = round(df['Lat'].max() + margin, 2)
south = round(df['Lat'].min() - margin, 2)
east = round(df['Lon'].max() + margin, 2)
west = round(df['Lon'].min() - margin, 2)

print(f"Bounding box: North = {north}, South = {south}, East = {east}, West = {west}")

years = df['Time'].dt.strftime('%Y').unique().tolist()
print("Years in the dataset:", years)

Bounding box: North = 61.73, South = 50.54, East = 11.7, West = -2.62
Years in the dataset: ['2023', '2024', '2025', '2026']


In [ ]:
URL = "https://cds.climate.copernicus.eu/api"
KEY = "KEY" 

client = cdsapi.Client(url=URL, key=KEY)

bounding_box = [61.73, -2.62, 50.54, 11.7]

years = ['2023', '2024', '2025', '2026']

all_months = [str(i).zfill(2) for i in range(1, 13)]
all_days = [str(i).zfill(2) for i in range(1, 32)]
all_hours = [f"{str(i).zfill(2)}:00" for i in range(24)]


for year in years:
    print(f"--- YEAR {year} ---")
    
    # wind (u, v)
    wind_nc = f"wind_{year}.nc"

    wind = {
        "product_type": ["reanalysis"],
        "variable": ["10m_u_component_of_wind", "10m_v_component_of_wind"],
        "year": [year],
        "month": all_months,
        "day": all_days,
        "time": all_hours,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": bounding_box
    }
    client.retrieve("reanalysis-era5-single-levels", wind, wind_nc)
    

    # waves (H_s)
    waves_nc = f"waves_{year}.nc"

    waves = {
        "product_type": ["reanalysis"],
        "variable": ["significant_height_of_combined_wind_waves_and_swell"],
        "year": [year],
        "month": all_months,
        "day": all_days,
        "time": all_hours,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": bounding_box
    }   
    client.retrieve("reanalysis-era5-single-levels", waves, waves_nc)


--- YEAR 2023 ---


2026-04-12 21:31:29,424 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-12 21:31:29,425 INFO Request ID is 0c3f1b73-b024-4ba8-8561-68d470d5a912
2026-04-12 21:31:29,494 INFO status has been updated to accepted
2026-04-12 21:31:43,176 INFO status has been updated to running
2026-04-12 21:43:52,453 INFO status has been updated to successful
2026-04-12 21:44:35,325 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2024 ---


2026-04-12 21:51:31,020 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-12 21:51:31,020 INFO Request ID is f5e65a8c-dc32-4998-a6e8-9f8f4dfb60f9
2026-04-12 21:51:31,106 INFO status has been updated to accepted
2026-04-12 21:51:39,737 INFO status has been updated to running
2026-04-12 22:03:53,488 INFO status has been updated to successful
2026-04-12 22:04:21,225 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2025 ---


2026-04-12 22:10:46,053 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-12 22:10:46,054 INFO Request ID is 4cabdf54-4cbe-44c0-995b-1909537a258c
2026-04-12 22:10:46,129 INFO status has been updated to accepted
2026-04-12 22:10:59,814 INFO status has been updated to running
2026-04-12 22:21:08,050 INFO status has been updated to successful
2026-04-12 22:21:43,550 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2026 ---


2026-04-12 22:28:09,582 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-12 22:28:09,582 INFO Request ID is 6c86ec9b-f64a-41f4-9580-827c00c7e0cb
2026-04-12 22:28:09,654 INFO status has been updated to accepted
2026-04-12 22:28:23,215 INFO status has been updated to running
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/jobs/6c86ec9b-f64a-41f4-9580-827c00c7e0cb?log=True&request=True (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001

Taking into account additional variables from ERA5:

In [ ]:

URL = "https://cds.climate.copernicus.eu/api"
KEY = "KEY" 

client = cdsapi.Client(url=URL, key=KEY)

bounding_box = [61.73, -2.62, 50.54, 11.7]
years = ['2023', '2024', '2025', '2026']

all_months = [str(i).zfill(2) for i in range(1, 13)]
all_days = [str(i).zfill(2) for i in range(1, 32)]
all_hours = [f"{str(i).zfill(2)}:00" for i in range(24)]


for year in years:
    print(f"--- YEAR {year} ---")
    
    # additional wave parameters (direction and period)
    wave_extra_nc = f"wave_ext_{year}.nc"
    wave_extra = {
        "product_type": ["reanalysis"],
        "variable": [
            "mean_wave_direction", 
            "mean_wave_period"
        ],
        "year": [year],
        "month": all_months,
        "day": all_days,
        "time": all_hours,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": bounding_box
    }
    
    client.retrieve("reanalysis-era5-single-levels", wave_extra, wave_extra_nc)

    # areodynamics (air density and gusts)
    wind_extra_nc = f"wind_extra_{year}.nc"
    wind_extra = {
        "product_type": ["reanalysis"],
        "variable": [
            "air_density_over_the_oceans", 
            "10m_wind_gust_since_previous_post_processing"
        ],
        "year": [year],
        "month": all_months,
        "day": all_days,
        "time": all_hours,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": bounding_box
    }
    
    client.retrieve("reanalysis-era5-single-levels", wind_extra, wind_extra_nc)


--- YEAR 2023 ---


2026-04-13 17:13:17,168 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-13 17:13:17,171 INFO Request ID is c3646484-6b1b-401b-a654-5cd50e68520e
2026-04-13 17:13:17,255 INFO status has been updated to accepted
2026-04-13 17:13:31,033 INFO status has been updated to running
2026-04-13 17:23:40,117 INFO status has been updated to successful
2026-04-13 17:23:45,663 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2024 ---


2026-04-13 17:44:32,820 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-13 17:44:32,821 INFO Request ID is 6fe872b0-3db6-4666-a2aa-384b173e028a
2026-04-13 17:44:32,933 INFO status has been updated to accepted
2026-04-13 17:44:54,209 INFO status has been updated to running
2026-04-13 17:56:55,630 INFO status has been updated to successful
2026-04-13 17:57:01,494 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2025 ---


2026-04-13 18:17:56,961 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-13 18:17:56,962 INFO Request ID is 76e79e47-2507-4ba3-b3d4-2f77a2e4f05b
2026-04-13 18:17:57,036 INFO status has been updated to accepted
2026-04-13 18:18:10,602 INFO status has been updated to running
2026-04-13 18:28:18,698 INFO status has been updated to successful
2026-04-13 18:28:22,978 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

--- YEAR 2026 ---


2026-04-13 18:48:58,151 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-13 18:48:58,152 INFO Request ID is 820d6565-e3a6-46dd-ac8f-ab820ccd7a42
2026-04-13 18:48:58,226 INFO status has been updated to accepted
2026-04-13 18:49:11,753 INFO status has been updated to running
2026-04-13 18:53:17,826 INFO status has been updated to successful
2026-04-13 18:53:20,206 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-se

In [6]:
pip install copernicusmarine


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Obtaining dependency information for copernicusmarine from https://files.pythonhosted.org/packages/e8/b7/4a3e7885c524cc1da667c414e97ac12a81051596146bec99e0113429f759/copernicusmarine-2.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for arcosparse<0.5.0,>=0.4.2 from https://files.pythonhosted.org/packages/34/ec/c764296b694b425853baf444d9e4643949d2e16479f5b192f47fa24ba1fe/arcosparse-0.4.2-py3-none-any.whl.metadata
  Obtaining dependency information for pydantic<3.0.0,>=2.9.1 from https://files.pythonhosted.org/packages/01/d7/c3a52c61f5b7be648e919005820fbac33028c6149994cd64453f49951c17/pydantic-2.13.0-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/107.9 kB ? eta -:--:--
     --- ------------------------------------ 10.2/107.9 kB ? eta -:--:--
     -------------- ---------------------- 41.0/107.9 kB 487.6 kB/s eta 0:00:01
     ------------------------------------ 107.9/107.9 kB 887.4 kB/s eta 0:00:00
  Obtaining dependency information for p

In [ ]:
copernicusmarine.login()

INFO - 2026-04-13T18:59:09Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:Copernicus Marine password:

INFO - 2026-04-13T18:59:46Z - Credentials file stored in C:\Users\aleks\.copernicusmarine\.copernicusmarine-credentials.


True

In [12]:
bounding_box = [61.73, -2.62, 50.54, 11.7]
years = ['2023', '2024', '2025', '2026']


for year in years:
    print(f"--- YEAR {year} ---")

    tides_nc = f"tides_{year}.nc"
    
    copernicusmarine.subset(
        dataset_id="cmems_mod_glo_phy_anfc_0.083deg_PT1H-m", # PT1H - hourly data
        variables=["uo", "vo"],
        minimum_latitude=bounding_box[2], # S
        maximum_latitude=bounding_box[0], # N
        minimum_longitude=bounding_box[1], # W
        maximum_longitude=bounding_box[3], # E
        start_datetime=f"{year}-01-01T00:00:00",
        end_datetime=f"{year}-12-31T23:59:59",
        output_filename=tides_nc,
        force_download=True
    )

WARNING - 2026-04-13T19:11:01Z - 'force_download' has been deprecated.


--- YEAR 2023 ---


INFO - 2026-04-13T19:11:03Z - Selected dataset version: "202406"
INFO - 2026-04-13T19:11:03Z - Selected dataset part: "default"
100%|██████████| 234/234 [04:11<00:00,  1.08s/it]
WARNING - 2026-04-13T19:15:18Z - 'force_download' has been deprecated.


--- YEAR 2024 ---


INFO - 2026-04-13T19:15:21Z - Selected dataset version: "202406"
INFO - 2026-04-13T19:15:21Z - Selected dataset part: "default"
100%|██████████| 234/234 [04:12<00:00,  1.08s/it]
WARNING - 2026-04-13T19:19:35Z - 'force_download' has been deprecated.


--- YEAR 2025 ---


INFO - 2026-04-13T19:19:37Z - Selected dataset version: "202406"
INFO - 2026-04-13T19:19:37Z - Selected dataset part: "default"
100%|██████████| 198/198 [03:08<00:00,  1.05it/s]
WARNING - 2026-04-13T19:22:48Z - 'force_download' has been deprecated.


--- YEAR 2026 ---


INFO - 2026-04-13T19:22:50Z - Selected dataset version: "202406"
INFO - 2026-04-13T19:22:50Z - Selected dataset part: "default"
WARNING - 2026-04-13T19:22:50Z - Some of your subset selection [2026-01-01 00:00:00+00:00, 2026-12-31 23:59:59+00:00] for the time dimension exceed the dataset coordinates [2022-06-01 00:00:00+00:00, 2026-04-22 23:00:00+00:00]
100%|██████████| 82/82 [01:46<00:00,  1.30s/it]
